#RQ1: Role-Salary Prediction
**Dissertation:** AI Approaches to Analysing Recruitment Demand: Machine learning insights from European Pharmaceutical Job Postings  
**Author:** Kashmira Bhoir  
**Institution:** Gisma University of Applied Sciences  
**Date:** 21 June 2026  

## Research Question
To what extent can multi-task transformers accurately predict
pharmaceutical job roles and associated salary levels from job posting text?

## Overview
This notebook helps to build a dual-head multi-task transformer that together:
- Head 1: Classifies pharma job roles
- Head 2: Predicts associated salaries by level (€)

## Key Findings
- Role Classification Accuracy : 90.4%
- Overall Salary MAE           : €9,627
- Best per-role MAE            : €7,356 (Pharmaceutical,Healthcare And Medical Sales)

## Input
- `Europe_Pharma_Jobs_Cleaned.xlsx` — 9,520 rows, 16 columns

## Outcome
- Trained dual-head Keras model
- Role classification report (precision,F1, recall)
- Salary prediction for each role

## Research Gap Addressed
Ather et al. (2024) and Agrawal et al. (2025) focused exclusively on US AI/ML job market. This is the first multi-task transformer for
joint pharmaceutical role + salary prediction on European job posting data.

Install and Import Libraries

In [26]:
!pip install sentence-transformers tensorflow scikit-learn pandas numpy -q

import pandas as pd
import numpy as np
import random
import re
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (accuracy_score, mean_absolute_error,
                             classification_report)
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

print(f"Libraries loaded")

Libraries loaded


Set Random Seeds for Reproducibility

In [27]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

Loading the cleaned dataset

In [28]:
file_path = "/content/Europe_Pharma_Jobs_Cleaned.xlsx"

df = pd.read_excel(file_path)
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

print(f"Loaded: {df.shape[0]:,} rows  |  {df.shape[1]} columns")
print(f"\n   Columns: {df.columns.tolist()}")

Loaded: 9,520 rows  |  16 columns

   Columns: ['data_source', 'category', 'company_name', 'job_description', 'job_title', 'job_type', 'location', 'post_date', 'salary_offered', 'post_year', 'salary_min_eur', 'salary_max_eur', 'currency_detected', 'salary_mid_eur', 'job_type_clean', 'country']


Targetting available Salary check

In [29]:
parseable = df['salary_mid_eur'].notna().sum()
print(f"Salary available")
print(f"   Parseable : {parseable:,}  ({parseable/len(df)*100:.1f}%)")
print(f"   Hidden    : {len(df)-parseable:,}  ({(len(df)-parseable)/len(df)*100:.1f}%)")
print(f"\n   Salary stats (EUR):")
print(df['salary_mid_eur'].describe().apply(lambda x: f"   {x:,.0f}").to_string())

Salary available
   Parseable : 1,680  (17.6%)
   Hidden    : 7,840  (82.4%)

   Salary stats (EUR):
count         1,680
mean         69,374
std          36,400
min          10,170
25%          43,183
50%          59,661
75%          85,058
max         261,372


Listing down all categories and selecting the top 5

In [30]:
category_counts = df.loc[df['salary_mid_eur'].notna(), 'category'].value_counts()

print(f"All categories with at least 1 salary-labeled row ({len(category_counts)} total):")
print(category_counts.to_string())

KEEP_ROLES = category_counts.head(5).index.tolist()

print(f"\nTop 5 by count - selected as KEEP_ROLES:")
print(category_counts.head(5).to_string())

df['role_label'] = df['category'].apply(lambda x: x if x in KEEP_ROLES else None)
df_joint = df[df['role_label'].notna() & df['salary_mid_eur'].notna()].copy()

print(f"\nRole categories (top 5):")
rc = df_joint['role_label'].value_counts().reset_index()
rc.columns = ['Role', 'Count']
rc['Share %'] = (rc['Count'] / len(df_joint) * 100).round(1)
print(rc.to_string(index=False))
print(f"\n   Total joint rows: {len(df_joint):,}")

All categories with at least 1 salary-labeled row (48 total):
category
Pharmaceutical, Healthcare And Medical Sales     567
Manufacturing & Operations                       207
Clinical Research                                167
Pharmaceutical Marketing                         140
Regulatory Affairs                               118
Science                                           92
Data Management And Statistics                    60
Quality-Assurance                                 50
Uncategorized                                     48
Healthcare                                        41
Forschende Pharmazeutische Industrie              40
Medical Information And Pharmacovigilance         25
Pharmazeutische Industrie                         22
Informationstechnologie                           18
√Ñffentliches Gesundheitswesen                    13
Medizintechnik                                     8
Medizintechnische Ger√§Te                          7
Medical Affairs / Pharmaceut

Building Combined Text Input for Embeddings

In [31]:
def build_text(row):
    return (f"{row.get('job_title','')}. "
            f"{row.get('country','')}. "
            f"{row.get('job_type_clean','')}. "
            f"{str(row.get('job_description',''))[:300]}")

df_joint['text_input'] = df_joint.apply(build_text, axis=1)

print(f"Text input built for {len(df_joint):,} rows")
print(f"\n   Sample text input:")
print(f"   {df_joint['text_input'].iloc[0][:200]}...")

Text input built for 1,199 rows

   Sample text input:
   Quality Engineer. Germany. Full-Time.  Quality Engineer-World Leading Medical Device Company-Germany
A new opportunity with a world leading medical device company has come in for a QMS and quality spe...


Generating Sentence Embeddings

In [32]:
print(f" Loading sentence-transformer model...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

print(f" Generating embeddings for {len(df_joint):,} rows...")
embeddings = embedder.encode(
    df_joint['text_input'].tolist(),
    batch_size        = 64,
    show_progress_bar = True
)

print(f"\nEmbeddings shape: {embeddings.shape}")
print(f"   Each job posting → {embeddings.shape[1]}-dimensional vector")

 Loading sentence-transformer model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

 Generating embeddings for 1,199 rows...


Batches:   0%|          | 0/19 [00:00<?, ?it/s]


Embeddings shape: (1199, 384)
   Each job posting → 384-dimensional vector


Encoding job role labels and Standardising Salary for Model Training

In [33]:
le_role   = LabelEncoder()
y_role    = le_role.fit_transform(df_joint['role_label'])
n_classes = len(le_role.classes_)

y_salary   = df_joint['salary_mid_eur'].values
sal_mean   = y_salary.mean()
sal_std    = y_salary.std()
y_sal_norm = (y_salary - sal_mean) / sal_std

print(f"Labels encoded")
print(f"   Role classes ({n_classes}): {list(le_role.classes_)}")
print(f"\n   Salary normalisation:")
print(f"   Mean : €{sal_mean:,.0f}")
print(f"   Std  : €{sal_std:,.0f}")
print(f"   Range: {y_sal_norm.min():.2f} → {y_sal_norm.max():.2f}")

Labels encoded
   Role classes (5): ['Clinical Research', 'Manufacturing & Operations', 'Pharmaceutical Marketing', 'Pharmaceutical, Healthcare And Medical Sales', 'Regulatory Affairs']

   Salary normalisation:
   Mean : €62,079
   Std  : €32,094
   Range: -1.62 → 6.21


Splitting Data into Train and Test Sets

In [34]:
X_train, X_test, \
yr_train, yr_test, \
ys_train, ys_test = train_test_split(
    embeddings, y_role, y_sal_norm,
    test_size    = 0.2,
    random_state = 42,
    stratify     = y_role
)

print(f"Train/Test split:")
print(f"   Train rows : {len(X_train):,}")
print(f"   Test rows  : {len(X_test):,}")
print(f"\n   Class distribution (train):")
for i, cls in enumerate(le_role.classes_):
    count = (yr_train == i).sum()
    print(f"   {cls:<45} : {count:,}")

Train/Test split:
   Train rows : 959
   Test rows  : 240

   Class distribution (train):
   Clinical Research                             : 134
   Manufacturing & Operations                    : 166
   Pharmaceutical Marketing                      : 112
   Pharmaceutical, Healthcare And Medical Sales  : 453
   Regulatory Affairs                            : 94


Building Dual-Head Multi-Task Model

In [35]:
inputs = keras.Input(shape=(384,), name='embedding_input')

shared = layers.Dense(256, activation='relu')(inputs)
shared = layers.BatchNormalization()(shared)
shared = layers.Dropout(0.3)(shared)
shared = layers.Dense(128, activation='relu')(shared)
shared = layers.BatchNormalization()(shared)
shared = layers.Dropout(0.3)(shared)

role_h   = layers.Dense(64, activation='relu')(shared)
role_h   = layers.Dropout(0.3)(role_h)
role_out = layers.Dense(
    n_classes, activation='softmax', name='role_output'
)(role_h)

sal_h   = layers.Dense(64, activation='relu')(shared)
sal_h   = layers.Dense(32, activation='relu')(sal_h)
sal_out = layers.Dense(
    1, activation='linear', name='salary_output'
)(sal_h)

model = keras.Model(inputs=inputs, outputs=[role_out, sal_out])

model.compile(
    optimizer    = keras.optimizers.Adam(learning_rate=3e-4),
    loss         = {
        'role_output'  : 'sparse_categorical_crossentropy',
        'salary_output': 'mse'
    },
    loss_weights = {'role_output': 1.0, 'salary_output': 1.0},
    metrics      = {
        'role_output'  : 'accuracy',
        'salary_output': 'mae'
    }
)

print(f"Model built — {model.count_params():,} total parameters")
model.summary()

Model built — 151,942 total parameters


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ embedding_input     │ (None, 384)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 256)       │     98,560 │ embedding_input[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256)       │      1,024 │ dense_5[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 256)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 128)       │     32,896 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ dense_6[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 128)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 64)        │      8,256 │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 64)        │      8,256 │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 64)        │          0 │ dense_7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 32)        │      2,080 │ dense_8[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ role_output (Dense) │ (None, 5)         │        325 │ dropout_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ salary_output       │ (None, 1)         │         33 │ dense_9[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 151,942 (593.52 KB)

 Trainable params: 151,174 (590.52 KB)

 Non-trainable params: 768 (3.00 KB)

Training the Multi-Task Model

In [36]:
print(f"  TRAINING — MULTITASK TRANSFORMER")

history = model.fit(
    X_train,
    {'role_output': yr_train, 'salary_output': ys_train},
    validation_data = (
        X_test,
        {'role_output': yr_test, 'salary_output': ys_test}
    ),
    epochs     = 60,
    batch_size = 32,
    callbacks  = [
        keras.callbacks.EarlyStopping(
            monitor              = 'val_loss',
            patience             = 10,
            restore_best_weights = True,
            verbose              = 1,
            mode                 = 'min'
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor  = 'val_loss',
            factor   = 0.5,
            patience = 5,
            verbose  = 1,
            min_lr   = 1e-6
        )
    ],
    verbose = 1
)

  TRAINING — MULTITASK TRANSFORMER
Epoch 1/60
30/30 ━━━━━━━━━━━━━━━━━━━━ 5s 28ms/step - loss: 3.4609 - role_output_accuracy: 0.2826 - role_output_loss: 2.0395 - salary_output_loss: 1.4202 - salary_output_mae: 0.8380 - val_loss: 2.4864 - val_role_output_accuracy: 0.5167 - val_role_output_loss: 1.5424 - val_salary_output_loss: 0.9352 - val_salary_output_mae: 0.7323 - learning_rate: 3.0000e-04
Epoch 2/60
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 2.3891 - role_output_accuracy: 0.4442 - role_output_loss: 1.4296 - salary_output_loss: 0.9588 - salary_output_mae: 0.7217 - val_loss: 2.4378 - val_role_output_accuracy: 0.4792 - val_role_output_loss: 1.4794 - val_salary_output_loss: 0.9487 - val_salary_output_mae: 0.7545 - learning_rate: 3.0000e-04
Epoch 3/60
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 2.0033 - role_output_accuracy: 0.5527 - role_output_loss: 1.2078 - salary_output_loss: 0.7954 - salary_output_mae: 0.6439 - val_loss: 2.3829 - val_role_output_accuracy: 0.4792 - val_role_o

Evaluating Model on Test Set

In [37]:
role_probs, sal_preds_norm = model.predict(X_test, verbose=0)

role_preds = np.argmax(role_probs, axis=1)
accuracy   = accuracy_score(yr_test, role_preds) * 100
sal_preds_eur   = sal_preds_norm.flatten() * sal_std + sal_mean
sal_actuals_eur = ys_test * sal_std + sal_mean
mae             = mean_absolute_error(sal_actuals_eur, sal_preds_eur)

print(f"  EVALUATION RESULTS")

print(f"\n Role Accuracy : {accuracy:.1f}%")
print(f" Salary MAE   : €{mae:,.0f}")

print(f"\n  Per-role classification report:")
print(classification_report(yr_test, role_preds,
                             target_names=le_role.classes_))

  EVALUATION RESULTS

 Role Accuracy : 90.4%
 Salary MAE   : €9,627

  Per-role classification report:
                                              precision    recall  f1-score   support

                           Clinical Research       0.82      0.85      0.84        33
                  Manufacturing & Operations       0.86      0.88      0.87        41
                    Pharmaceutical Marketing       0.86      0.86      0.86        28
Pharmaceutical, Healthcare And Medical Sales       0.96      0.95      0.96       114
                          Regulatory Affairs       0.88      0.88      0.88        24

                                    accuracy                           0.90       240
                                   macro avg       0.88      0.88      0.88       240
                                weighted avg       0.91      0.90      0.90       240



Building Role-to-Salary Mapping Table

In [38]:
df_results = pd.DataFrame({
    'role_actual'     : le_role.inverse_transform(yr_test),
    'role_predicted'  : le_role.inverse_transform(role_preds),
    'salary_actual'   : sal_actuals_eur,
    'salary_predicted': sal_preds_eur
})

summary = df_results.groupby('role_actual').agg(
    count        = ('salary_actual', 'count'),
    avg_actual   = ('salary_actual', 'mean'),
    avg_pred     = ('salary_predicted', 'mean'),
    mae_per_role = ('salary_actual',
        lambda x: mean_absolute_error(
            x, df_results.loc[x.index, 'salary_predicted']))
).reset_index()

summary.columns            = ['Role','Count','Avg Actual €',
                               'Avg Predicted €','MAE €']
summary['Avg Actual €']    = summary['Avg Actual €'].round(0).astype(int)
summary['Avg Predicted €'] = summary['Avg Predicted €'].round(0).astype(int)
summary['MAE €']           = summary['MAE €'].round(0).astype(int)

print(f"  ROLE : SALARY MAPPING")
print(f"\n  {'Role':<45} {'N':>5} {'Actual €':>10} "
      f"{'Pred €':>10} {'MAE €':>8}")

for _, row in summary.iterrows():
    print(f"  {row['Role']:<45} {row['Count']:>5} "
          f"{row['Avg Actual €']:>10,} "
          f"{row['Avg Predicted €']:>10,} "
          f"{row['MAE €']:>8,}")

  ROLE : SALARY MAPPING

  Role                                              N   Actual €     Pred €    MAE €
  Clinical Research                                33     80,703     74,660   13,435
  Manufacturing & Operations                       41     66,969     70,138   13,392
  Pharmaceutical Marketing                         28     50,825     49,593    8,030
  Pharmaceutical, Healthcare And Medical Sales    114     52,352     52,669    7,356
  Regulatory Affairs                               24     73,488     73,907   10,612


Show Final Training History

In [39]:
print(f"  TRAINING HISTORY (last 5 epochs)")

print(f"\n  {'Epoch':>6} {'Loss':>10} {'Train Acc':>10} "
      f"{'Val Loss':>10} {'Val Acc':>10}")


total_ep = len(history.history['loss'])
for i in range(max(0, total_ep - 5), total_ep):
    print(f"  {i+1:>6} "
          f"{history.history['loss'][i]:>10.4f} "
          f"{history.history['role_output_accuracy'][i]*100:>9.1f}% "
          f"{history.history['val_loss'][i]:>10.4f} "
          f"{history.history['val_role_output_accuracy'][i]*100:>9.1f}%")

  TRAINING HISTORY (last 5 epochs)

   Epoch       Loss  Train Acc   Val Loss    Val Acc
      40     0.4634      89.6%     0.5295      87.5%
      41     0.4196      90.7%     0.5267      87.9%
      42     0.4227      90.0%     0.5282      87.5%
      43     0.4289      90.3%     0.5337      87.9%
      44     0.3707      91.2%     0.5381      88.3%


In [40]:
clinical     = summary[summary['Role'].str.contains(
    'Clinical', case=False, na=False)]
clin_sal     = int(clinical['Avg Predicted €'].values[0]) \
               if len(clinical) > 0 else int(sal_mean)
clin_mae     = int(clinical['MAE €'].values[0]) \
               if len(clinical) > 0 else int(mae)
best_row     = summary.sort_values('MAE €').iloc[0]

TL="\u2554"; TR="\u2557"; BL="\u255A"; BR="\u255D"
LT="\u2560"; RT="\u2563"; H="\u2550";  V="\u2551"; W=61

def row(text=""): return V + text.ljust(W) + V

print(TL + H*W + TR)
print(row())
print(row("     KEY RESEARCH FINDING — RQ1"))
print(row("   Role-Salary Patterns — European Pharma Market"))
print(row())
print(LT + H*W + RT)
print(row())
print(row("   Clinical Research roles predict"))
print(row())
print(row(f"       Avg Predicted Salary : {clin_sal:,}"))
print(row(f"       Role MAE : {clin_mae:,}"))
print(row(f"       Overall Accuracy : {accuracy:.1f}%"))
print(row(f"       Overall Salary MAE  : {int(mae):,}"))
print(row())
print(LT + H*W + RT)
print(row())
print(row(f"   Best per-role MAE:"))
print(row(f"       {best_row['Role']} : {best_row['MAE €']:,}"))
print(row())
print(LT + H*W + RT)
print(row())
print(row("   Gap addressed:"))
print(row("   Ather (2024) + Agrawal (2025) — US AI/ML market only"))
print(row("   \u2192 First EU pharma multi-task transformer"))
print(row())
print(BL + H*W + BR)

╔═════════════════════════════════════════════════════════════╗
║                                                             ║
║     KEY RESEARCH FINDING — RQ1                              ║
║   Role-Salary Patterns — European Pharma Market             ║
║                                                             ║
╠═════════════════════════════════════════════════════════════╣
║                                                             ║
║   Clinical Research roles predict                           ║
║                                                             ║
║       Avg Predicted Salary : 74,660                         ║
║       Role MAE : 13,435                                     ║
║       Overall Accuracy : 90.4%                              ║
║       Overall Salary MAE  : 9,627                           ║
║                                                             ║
╠═════════════════════════════════════════════════════════════╣
║                                       